# 10 - Boosting on TF-IDF

In [20]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import accuracy_score, classification_report

from itertools import product

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

In [ ]:
df = pd.read_csv('../data/processed/combined_base_data.csv')
df['datetime_posted'] = pd.to_datetime(df['datetime_posted'], utc=True, format='mixed')
df = df.dropna(subset=['title']).copy()

print(f'Total articles: {len(df)}')
print(f'Fox: {df["is_fox"].sum()} | NBC: {(df["is_fox"]==0).sum()}')
print(f'Missing datetimes: {df["datetime_posted"].isna().sum()}')

Total articles: 3,801
Fox: 2,000 | NBC: 1,801
Missing datetimes: 90


In [14]:
df_dated   = df.dropna(subset=['datetime_posted']).sort_values('datetime_posted')
df_undated = df[df['datetime_posted'].isna()]

split_idx = int(len(df_dated) * 0.8)
train_t = pd.concat([df_dated.iloc[:split_idx], df_undated], ignore_index=True)
test_t = df_dated.iloc[split_idx:].reset_index(drop=True)

print(f'\nTemporal split:')
print(f'\tTrain: {len(train_t)} | Fox%: {train_t["is_fox"].mean()}')
print(f'\tTest:  {len(test_t)}  | Fox%: {test_t["is_fox"].mean()}')
print(f'\tTest date range: {test_t["datetime_posted"].min().date()} → {test_t["datetime_posted"].max().date()}')

X_train_text = train_t['title'].tolist()
X_test_text = test_t['title'].tolist()
y_train = train_t['is_fox'].tolist()
y_test = test_t['is_fox'].tolist()


Temporal split:
	Train: 3058 | Fox%: 0.5222367560497057
	Test:  743  | Fox%: 0.5423956931359354
	Test date range: 2024-09-27 → 2026-04-07


# TF-IDF Baselines

Start with the same cheap text pipeline across a few linear models, then compare them against tree-based and boosting variants.


In [15]:
vec_base = TfidfVectorizer(stop_words='english', max_features=100)
lr_base  = LogisticRegression(max_iter=100, random_state=42)
lr_base.fit(vec_base.fit_transform(X_train_text), y_train)
acc_base = accuracy_score(y_test, lr_base.predict(vec_base.transform(X_test_text)))
print(f'Assignment baseline (temporal): {acc_base}')

Assignment baseline (temporal): 0.6864064602960969


In [16]:
vec_imp  = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2), sublinear_tf=True)
lr_imp   = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_imp.fit(vec_imp.fit_transform(X_train_text), y_train)
acc_imp  = accuracy_score(y_test, lr_imp.predict(vec_imp.transform(X_test_text)))
print(f'Improved TF-IDF LR (temporal):  {acc_imp}')
print(classification_report(y_test, lr_imp.predict(vec_imp.transform(X_test_text)), target_names=['NBC', 'Fox']))

Improved TF-IDF LR (temporal):  0.7362045760430687
              precision    recall  f1-score   support

         NBC       0.72      0.69      0.70       340
         Fox       0.75      0.78      0.76       403

    accuracy                           0.74       743
   macro avg       0.73      0.73      0.73       743
weighted avg       0.74      0.74      0.74       743



/Users/parthas/Documents/cis5190_final/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/parthas/Documents/cis5190_final/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/Users/parthas/Documents/cis5190_final/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


In [19]:
from xgboost import XGBClassifier

# test a single XGB attempt
vec = TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1,2), sublinear_tf=True)
X_tr_vec = vec.fit_transform(X_train_text)
X_te_vec = vec.transform(X_test_text)

clf = XGBClassifier(random_state=42)
clf.fit(X_tr_vec, y_train)
acc_xgb = accuracy_score(y_test, clf.predict(X_te_vec))
print(f"TF-IDF + XGB - Temporal: {acc_xgb}")

TF-IDF + XGB - Temporal: 0.7012113055181696


In [ ]:
tfidf_params = {
    'max_features': [1000, 3000, 5000, 7000, 10000],
    'ngram_range': [(1,1), (1,2), (1,3), (1,4)]
}

xgb_params = {
    'n_estimators': [100, 300, 500, 700, 1000],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [1, 2, 3, 4, 5, 6],
    'subsample': [0.6, 0.8, 1.0]
}

tfidf_combos = list(product(tfidf_params['max_features'], tfidf_params['ngram_range']))
xgb_combos = list(product(xgb_params['n_estimators'], xgb_params['learning_rate'], xgb_params['max_depth'], xgb_params['subsample']))

total_combos = len(tfidf_combos) * len(xgb_combos)
n_iter = 100 
print(f"Total possible combinations: {total_combos}")
print(f"Random iterations: {n_iter}")

rng = np.random.RandomState(42)
random_indices = rng.choice(total_combos, size=n_iter, replace=False)

results = []
for iteration, flat_idx in enumerate(random_indices):
    tfidf_idx = flat_idx // len(xgb_combos)
    xgb_idx = flat_idx % len(xgb_combos)
    
    max_feat, ngram = tfidf_combos[tfidf_idx]
    n_est, lr, depth, subsamp = xgb_combos[xgb_idx]
    
    vec = TfidfVectorizer(stop_words='english', max_features=max_feat, ngram_range=ngram, sublinear_tf=True)
    X_tr_vec = vec.fit_transform(X_train_text)
    X_te_vec = vec.transform(X_test_text)
    
    xgb_clf = XGBClassifier(n_estimators=n_est, learning_rate=lr, max_depth=depth,
                            subsample=subsamp, colsample_bytree=0.8, random_state=42, 
                            eval_metric='logloss')
    
    xgb_clf.fit(X_tr_vec, y_train)
    
    acc = accuracy_score(y_test, xgb_clf.predict(X_te_vec))
    
    results.append({
        'max_features': max_feat,
        'ngram_range': ngram,
        'n_estimators': n_est,
        'learning_rate': lr,
        'max_depth': depth,
        'subsample': subsamp,
        'accuracy': acc
    })
    
    current_best = max(r['accuracy'] for r in results)
    if iteration % 10 == 0:
        print(f"Current best: {current_best}")

results_df = pd.DataFrame(results).sort_values('accuracy', ascending=False)
print(f"Best configuration:\n{results_df.iloc[0].to_dict()}")
print(f"\nBest accuracy: {results_df['accuracy'].max()}")

Total possible combinations: 7200
Random iterations: 100
  Current best: 0.6702557200538358
  Current best: 0.7092866756393001
  Current best: 0.7146702557200538
  Current best: 0.7240915208613729
  Current best: 0.7240915208613729
  Current best: 0.7240915208613729
  Current best: 0.7240915208613729
  Current best: 0.7240915208613729
  Current best: 0.7240915208613729
  Current best: 0.7240915208613729
Best configuration:
{'max_features': 10000, 'ngram_range': (1, 1), 'n_estimators': 1000, 'learning_rate': 0.03, 'max_depth': 4, 'subsample': 0.8, 'accuracy': 0.7240915208613729}

Best accuracy: 0.7240915208613729
